In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime, timedelta


class IcebergSparkSession:
    """
    # 기본 설정으로 사용
    iceberg_session = IcebergSparkSession()
    spark = iceberg_session.get_spark()
    
    # 또는 with 문 사용
    with IcebergSparkSession(app_name="MyApp", s3_endpoint="http://minio2:9000") as spark:
        # 스파크 작업 수행
        df = spark.sql("SELECT 1")
        df.show()
        
    # 또는 사용자 정의 설정으로 사용
    custom_session = IcebergSparkSession(
        app_name="CustomApp",
        warehouse_path="s3a://my-warehouse/",
        s3_endpoint="http://custom-minio:9000",
        s3_access_key="mykey",
        s3_secret_key="mysecret",
        region="us-west-2"
    )
    spark = custom_session.get_spark()
    # 작업 후 종료
    custom_session.stop()
    """
    
    def __init__(self, app_name="IcebergTest", warehouse_path="s3a://warehouse/", 
                 s3_endpoint="http://minio1:9000", s3_access_key="admin", 
                 s3_secret_key="admin1234", region="us-east-1", install_packages=False):
        self.app_name = app_name
        self.warehouse_path = warehouse_path
        self.s3_endpoint = s3_endpoint
        self.s3_access_key = s3_access_key
        self.s3_secret_key = s3_secret_key
        self.region = region
        self.install_packages = install_packages
        self.spark = self.create_spark_session()
        self.configure_s3()

    def create_spark_session(self, install_packages=False):
        """
        SparkSession을 생성합니다.
        
        Args:
            install_packages (bool): 추가 패키지를 설치할지 여부. 기본값은 True입니다.
                                    False로 설정하면 패키지 설치를 건너뜁니다.
        
        Returns:
            SparkSession: 구성된 SparkSession 객체
        """
        # 기본 SparkSession 빌더 생성
        builder = SparkSession.builder.appName(self.app_name)
        
        # 패키지 설치가 활성화된 경우에만 패키지 설정 추가
        if install_packages:
            packages = [
                "org.apache.hadoop:hadoop-aws:3.3.4",
                'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.8.1',
                'com.amazonaws:aws-java-sdk-bundle:1.12.769'
            ]
            builder = builder.config("spark.jars.packages", ",".join(packages))
        
        # 공통 설정 추가
        builder = builder \
            .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
            .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
            .config("spark.sql.catalog.spark_catalog.type", "hive") \
            .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
            .config("spark.sql.catalog.local.type", "hadoop") \
            .config("spark.sql.catalog.local.warehouse", self.warehouse_path) \
            .config("spark.sql.catalog.local.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
            .config("spark.sql.defaultCatalog", "local") \
            .config("spark.driver.extraJavaOptions", f"-Daws.region={self.region} -Daws.s3.path.style.access=true") \
            .config("spark.executor.extraJavaOptions", f"-Daws.region={self.region} -Daws.s3.path.style.access=true")
    
        return builder.getOrCreate()
      

    def configure_s3(self):
        # S3 기본 설정
        self.spark.conf.set("spark.hadoop.fs.s3a.access.key", self.s3_access_key)
        self.spark.conf.set("spark.hadoop.fs.s3a.secret.key", self.s3_secret_key)
        self.spark.conf.set("spark.hadoop.fs.s3a.endpoint", self.s3_endpoint)
        self.spark.conf.set("spark.hadoop.fs.s3a.path.style.access", "true")
        self.spark.conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        self.spark.conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        self.spark.conf.set("spark.hadoop.fs.s3a.region", self.region)
        
        # S3 연결 설정
        self.spark.conf.set("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        self.spark.conf.set("spark.hadoop.fs.s3a.impl.disable.cache", "true")
        self.spark.conf.set("spark.hadoop.fs.s3a.debug.detailed.exceptions", "true")
        self.spark.conf.set("spark.hadoop.fs.s3a.bucket.probe", "0")
        self.spark.conf.set("spark.hadoop.fs.s3a.change.detection.mode", "none")
        
        # Iceberg S3 설정
        self.spark.conf.set("spark.sql.catalog.local.s3.region", self.region)
        self.spark.conf.set("spark.sql.catalog.local.s3.access-key-id", self.s3_access_key)
        self.spark.conf.set("spark.sql.catalog.local.s3.secret-access-key", self.s3_secret_key)
        self.spark.conf.set("spark.sql.catalog.local.s3.endpoint", self.s3_endpoint)
        self.spark.conf.set("spark.sql.catalog.local.s3.path-style-access", "true")
        
    def get_spark(self):
        """SparkSession 객체 반환"""
        return self.spark
    
    def stop(self):
        """SparkSession 종료"""
        if self.spark:
            self.spark.stop()
            
    def __enter__(self):
        """컨텍스트 매니저 지원"""
        return self.spark
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """컨텍스트 매니저 종료 시 SparkSession 종료"""
        self.stop()

In [2]:
session = IcebergSparkSession(
    app_name="CustomApp",
    warehouse_path="s3a://warehouse/",
    s3_endpoint="http://minio1:9000",
    s3_access_key="admin",
    s3_secret_key="admin1234",
    install_packages=False,
    region="us-east-1"
)

spark = session.get_spark()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/07 14:48:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:

# 테스트 데이터 생성
now = datetime.now()
data = [
    (3, "제품C", 150, now - timedelta(days=3)),
    (4, "제품D", 300, now - timedelta(days=2)),
    (5, "제품E", 250, now - timedelta(days=1)),
    (6, "제품F", 150, now - timedelta(days=3)),
    (7, "제품G", 300, now - timedelta(days=2)),
    (8, "제품H", 250, now - timedelta(days=1))
]

# 스키마 정의
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("price", IntegerType(), True),
    StructField("created_at", TimestampType(), True)
])

# 데이터프레임 생성
test_df = spark.createDataFrame(data, schema)

# 데이터프레임 확인
print("생성된 데이터프레임:")
test_df.show()

생성된 데이터프레임:


25/03/06 16:08:30 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[Stage 4:=========================================>              (55 + 13) / 75]

+---+-----+-----+--------------------+
| id| name|price|          created_at|
+---+-----+-----+--------------------+
|  3|제품C|  150|2025-03-03 16:08:...|
|  4|제품D|  300|2025-03-04 16:08:...|
|  5|제품E|  250|2025-03-05 16:08:...|
|  6|제품F|  150|2025-03-03 16:08:...|
|  7|제품G|  300|2025-03-04 16:08:...|
|  8|제품H|  250|2025-03-05 16:08:...|
+---+-----+-----+--------------------+



In [2]:
# Iceberg 테이블 이름 설정
table_name = "local.test_db.products"

# 기존 테이블이 있으면 삭제 (테스트 목적)
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

 # 테이블 생성 (SQL 방식)
spark.sql(f"""
   CREATE TABLE IF NOT EXISTS {table_name} (
     id INT,
     name STRING,
     price INT,
     created_at TIMESTAMP
   ) USING iceberg
""")

# spark.sql("""CREATE TABLE IF NOT EXISTS iceberg.table1 (
# revenue int,
# department string,
# boss string)
# USING iceberg
# location 's3://ds.iceberg/warehouse/table1'""")

print(f"테이블 '{table_name}'이 성공적으로 생성되었습니다.")

25/03/06 10:11:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


테이블 'local.test_db.products'이 성공적으로 생성되었습니다.


In [6]:
# Iceberg 테이블 이름 설정
table_name = "local.test_db.products"

# 임시 뷰로 등록
test_df.createOrReplaceTempView("updates")

# MERGE INTO 구문으로 upsert 수행
spark.sql(f"""
MERGE INTO {table_name} t
USING updates s
ON t.id = s.id
WHEN MATCHED THEN
  UPDATE SET 
    t.name = s.name,
    t.price = s.price,
    t.created_at = s.created_at
WHEN NOT MATCHED THEN
  INSERT (id, name, price, created_at)
  VALUES (s.id, s.name, s.price, s.created_at)
""")


DataFrame[]

In [4]:
# Iceberg 테이블 이름 설정
table_name = "local.test_db.products"
spark.table(table_name).show()

25/03/07 14:48:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[Stage 0:>                                                          (0 + 1) / 1]

+---+-----+-----+--------------------+
| id| name|price|          created_at|
+---+-----+-----+--------------------+
|  1|제품A|  100|2025-03-01 10:12:...|
|  2|제품B|  200|2025-03-02 10:12:...|
|  3|제품C|  150|2025-03-03 16:17:...|
|  4|제품D|  300|2025-03-04 16:17:...|
|  5|제품E|  250|2025-03-05 16:17:...|
|  6|제품F|  150|2025-03-03 16:17:...|
|  7|제품G|  300|2025-03-04 16:17:...|
|  8|제품H|  250|2025-03-05 16:17:...|
+---+-----+-----+--------------------+

